# Add Registrar's Notation for early AR filings (BAR AR Service)

<b> Purpose: Add Registrar's Notation filing to all businesses that used the BAR AR Service to file an Annual Report ahead of it being due.</b>

This is a one time (python) script to be run at a given date/time.<br>
Set the configuration (client_id, client_secret, url(s)) for a scpecific environment.<br>
Get access token for authorization.<br>

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import psycopg2

# this will load all the envars from a .env file located in the project root
load_dotenv(find_dotenv())

%load_ext sql

In [ ]:
connect_to_db = 'postgresql://' + \
                os.getenv('ENTITY_DATABASE_USERNAME', '') + ":" + os.getenv('ENTITY_DATABASE_PASSWORD', '') +'@' + \
                os.getenv('ENTITY_DATABASE_HOST', '') + ':' + os.getenv('ENTITY_DATABASE_PORT', '5432') + '/' + \
                os.getenv('ENTITY_DATABASE_NAME', '');
connect_to_db
    
%sql $connect_to_db

In [ ]:
%%sql 
select now() AT TIME ZONE 'PST' as current_date

In [ ]:
import requests
import os
from datetime import datetime

# token_url, client_id, client_secret, base_url - update based on environment
token_url = os.getenv('ACCOUNT_SVC_AUTH_URL')
client_id = os.getenv('ACCOUNT_SVC_CLIENT_ID')
client_secret = os.getenv('ACCOUNT_SVC_CLIENT_SECRET')
base_url = os.getenv('LEGAL_API_BASE_URL')

header = {
    "Content-Type": "application/x-www-form-urlencoded"
}

data = 'grant_type=client_credentials'

res = requests.post(token_url, data, auth=(client_id, client_secret), headers=header)

# Check the status code of the response
if res.status_code == 200:
    print(f"Access token returned successfully : {base_url}")
    token = res.json()["access_token"]
else:
    print(f"Failed to make POST request. Status code: {res.status_code}")
    print(res.text)  # Print the error message if the request fails

Call API (POST) endpoint to create Registrar's Notation filing for businesses.

In [ ]:
from urllib.parse import urljoin
from rn_early_ar_output import businesses

current_date = datetime.now().date().isoformat()
headers = {
    'Content-Type': 'application/json',
    'Authorization': 'Bearer ' + token
}

successful_identifiers = []
failed_identifiers = []
skipped_identifiers = []
duplicate_identifiers = []

# loop through list of businesses to create filing
for identifier in businesses:
    business_details = %sql \
        SELECT b.state, b.legal_type, \
            (SELECT COUNT(1) \
                FROM filings f \
                WHERE f.business_id = b.id \
                AND f.status in ('DRAFT', 'PENDING')) <> 0 AS has_draft_or_pending, \
            (SELECT COUNT(1) \
                FROM filings f \
                WHERE f.business_id = b.id \
                AND f.filing_json->'filing'->'header'->>'name' = 'registrarsNotation' \
                AND f.filing_json->'filing'->'registrarsNotation'->>'orderDetails' = 'Early AR Filed') <> 0 AS has_early_ar_filed \
        FROM businesses b \
        WHERE b.identifier = :identifier

    state = None
    legal_type = None
    has_draft_or_pending = None
    has_early_ar_filed = None

    if business_details:
        row = business_details[0]
        state = row._mapping['state']
        legal_type = row._mapping['legal_type']
        has_draft_or_pending = row._mapping['has_draft_or_pending']
        has_early_ar_filed = row._mapping['has_early_ar_filed']

    if has_early_ar_filed:
        duplicate_identifiers.append(identifier)
        continue

    if state == 'HISTORICAL' or has_draft_or_pending:
        skipped_identifiers.append(identifier)
        continue

    filing_data = {
        "filing": {
            "header": {
                "name": "registrarsNotation",
                "date": current_date,
                "certifiedBy": "system"
            },
            "business": {
                "identifier": identifier,
                "legalType": legal_type
            },
            "registrarsNotation": {
                "orderDetails": "Early AR Filed"
            }
        }
    }

    filing_url = urljoin(base_url, f"/api/v2/businesses/{identifier}/filings")
    response = requests.post(filing_url, headers=headers, json=filing_data)

    # Check the status code of the response
    if response.status_code == 201:
        successful_identifiers.append(identifier)
    else:
        failed_identifiers.append(identifier)
        print(f"Failed to make POST request. Status code: {response.status_code} for {identifier}")

print('Successfully filed Registrar Notation for:', successful_identifiers)
print('Failed to file Registrar Notation for:', failed_identifiers)
print('Skipped to file Registrar Notation for:', skipped_identifiers)
print('Already had Early AR Filed Registrar Notation:', duplicate_identifiers)